# Chapter 8: Code Execution

Companion notebook for *Build an AI Agent (From Scratch)*, Chapter 8.
Each listing in the book maps 1:1 to a cell below.

Many cells require `E2B_API_KEY` in `.env` to actually run. Cells that
rely on the sandbox are gated behind a comment so the notebook still
imports cleanly without an E2B account.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

%load_ext autoreload
%autoreload 2

## 8.1 Giving agents a computer

### 8.1.1 Limitations of predefined tool approaches

### 8.1.2 What code execution brings

Listings 8.1–8.4 are illustrative pseudocode showing different patterns
for combining tool calls with code. They reference helper functions
(`read_file`, `analyze_data`, `search_products`, `analyze_sentiment`,
etc.) that are not defined here, so they are not meant to be executed —
they show *what the agent could write* once code execution is enabled.

**Listing 8.1** Processing multiple files with a single code execution

In [ ]:
results = []
for i in range(1, 11):
    content = read_file(f"file_{i}.txt")
    results.append(content)

analysis = analyze_data(results)
print(f"Analysis result: {analysis}")

**Listing 8.2** Processing complex business logic with control flow

In [ ]:
budget_remaining = 5000
purchased_items = []

for product in search_products("laptop"):
    price = get_price(product)

    if price <= budget_remaining and product.rating >= 4.5:
        add_to_cart(product)
        budget_remaining -= price
        purchased_items.append(product)

**Listing 8.3** Efficient data transformation through intermediate variables

In [ ]:
raw_data = fetch_sales_data("2024-Q1")

cleaned_data = raw_data[raw_data['amount'] > 0]
grouped = cleaned_data.groupby('category').agg({
    'amount': 'sum',
    'quantity': 'count',
    'customer_id': 'nunique'
})

**Listing 8.4** Creating new tools through tool composition

In [ ]:
def analyze_company(company_name):
    """Simple company analysis combining news and stock data"""

    news = search_news(company_name, limit=10)
    stock_data = get_stock_price(company_name, days=30)

    summary = {
        'price_change': stock_data['change_percent'],
        'recent_news': [n['title'] for n in news[:3]],
        'sentiment': analyze_sentiment(news)
    }

    report = create_report(data=summary)

    return report

### 8.1.3 The effectiveness of code-based actions

## 8.2 Connecting code environments

### 8.2.1 Why sandboxes are necessary

### 8.2.2 Introduction to E2B and basic usage

Listings 8.5–8.8 use the E2B sandbox directly (without the agent).
They require `E2B_API_KEY` to run.

**Listing 8.5** Basic E2B sandbox usage flow

In [ ]:
from e2b_code_interpreter import Sandbox

# Create sandbox
sandbox = Sandbox.create()
print(f"Sandbox ID: {sandbox.id}")

# Execute code
code = """
result = 2 + 3
print(f"Calculation: 2 + 3 = {result}")
result
"""
execution = sandbox.run_code(code)

# Check results
print("Output:", execution.logs.stdout)
print("Return value:", execution.results[0].text)

# Terminate sandbox
sandbox.kill()

**Listing 8.6** Configuring timeout and environment variables

In [ ]:
import os

# 5-minute timeout, environment variable settings
sandbox = Sandbox.create(
    timeout=300,
    envs={
        'TAVILY_API_KEY': os.getenv('TAVILY_API_KEY'),
        'MY_CONFIG': 'some_value',
    }
)

# Extend timeout if needed
sandbox.set_timeout(600)  # Extend to 10 minutes

**Listing 8.7** State persistence across multiple code executions

In [ ]:
# First execution: define variable
sandbox.run_code("data = [1, 2, 3]; print('Data preparation complete')")

# Second execution: reuse variable
execution = sandbox.run_code("sum(data)")
print("Sum:", execution.results[0].text)  # 6

**Listing 8.8** Handling execution errors

In [ ]:
execution = sandbox.run_code("x = 1 / 0")
if execution.error:
    print("Error type:", execution.error.name)    # ZeroDivisionError
    print("Error message:", execution.error.value)  # division by zero

### 8.2.3 Connecting the sandbox to the agent

Listings 8.9–8.13 show diffs to `ExecutionContext`, the `Agent`
constructor and `_setup_tools`, the `run()` lifecycle, and the
`execute_python` tool itself. They are illustrative — the real,
merged versions live in `scratch_agent/context.py`,
`scratch_agent/agent.py`, and `scratch_agent/tools/code_execution.py`.

**Listing 8.9** Adding code_env field to ExecutionContext

In [ ]:
@dataclass
class ExecutionContext:
    ...
    code_env: Optional["Sandbox"] = None  # Added

**Listing 8.10** Adding code_execution parameter to Agent constructor

In [ ]:
class Agent:
    def __init__(
        ...
        code_execution: Optional[str] = None,  # Added
    ):
        self.code_execution = code_execution
        self.tools = self._setup_tools(tools or [])

**Listing 8.11** Auto-registering execute_python tool in _setup_tools

In [ ]:
def _setup_tools(self, tools: List[BaseTool]) -> List[BaseTool]:
    tools = list(tools)

    if self.code_execution == "e2b":
        from .tools.code_execution import execute_python
        tools.append(execute_python)

    return tools

**Listing 8.12** Managing sandbox lifecycle in the run method

In [ ]:
async def run(
    self,
    user_input: str,
    context: Optional[ExecutionContext] = None
) -> AgentResult:
    if context is None:
        context = ExecutionContext()

    # Initialize sandbox
    if self.code_execution == "e2b" and context.code_env is None:
        from e2b_code_interpreter import Sandbox
        context.code_env = Sandbox.create(timeout=300)

    try:
        # Existing agent execution logic
        ...
    finally:
        # Clean up sandbox
        if context.code_env is not None:
            context.code_env.kill()

**Listing 8.13** Implementing the execute_python tool

In [ ]:
@tool(
    name="execute_python",
    description="Execute Python code in a sandboxed environment. "
                "Use this to perform calculations, data processing, or any Python operations."
)
def execute_python(context: ExecutionContext, code: str) -> str:
    if context.code_env is None:
        raise RuntimeError("No code execution environment available.")

    sandbox: Sandbox = context.code_env
    execution = sandbox.run_code(code)

    return json.dumps(execution.to_json(), indent=2, ensure_ascii=False)

### 8.2.4 Testing the code execution agent

**Listing 8.14** Testing the code execution agent

*(Requires `E2B_API_KEY`. The book uses the package name `scratch_agents`
in imports; this companion code uses the singular `scratch_agent`
for installability.)*

In [ ]:
from scratch_agent import Agent, LlmClient

agent = Agent(
    model=LlmClient(model="gpt-5"),
    tools=[],  # No other tools
    instructions="You can execute Python code to solve problems.",
    code_execution="e2b"
)

# result = await agent.run("What is the 100th Fibonacci number?", verbose=True)
# print(result.output)

## 8.3 Porting tools to sandboxes

### 8.3.1 The need for tool portability

### 8.3.2 Extending FunctionTool

**Listing 8.15** Adding sandbox_executable flag to FunctionTool

In [ ]:
class FunctionTool(BaseTool):
    def __init__(
        ...
        sandbox_executable: bool = False,
    ):
        ...
        self.sandbox_executable = sandbox_executable

        # sandbox_executable tools cannot have context
        if sandbox_executable and self.needs_context:
            raise ValueError(
                f"Tool '{func.__name__}' cannot be sandbox_executable "
                "because it requires 'context' parameter."
            )

**Listing 8.16** Extracting function source code for sandbox registration

In [ ]:
def get_source_code(self) -> str:
    """Extract function source code for sandbox registration."""
    if not self.sandbox_executable:
        raise ValueError(f"Tool '{self.name}' is not marked as sandbox_executable")

    source = inspect.getsource(self.func)

    # Remove @tool decorator
    lines = source.split('\n')
    filtered_lines = []
    skip_decorator = False

    for line in lines:
        stripped = line.strip()
        if stripped.startswith('@tool'):
            skip_decorator = True
            # Check if single-line decorator
            if '(' not in stripped or ')' in stripped:
                skip_decorator = False
            continue
        if skip_decorator:
            if ')' in stripped:
                skip_decorator = False
            continue
        filtered_lines.append(line)

    return '\n'.join(filtered_lines)

### 8.3.3 Implementing the Wikipedia Revision tool

**Listing 8.17** Wikipedia revision tool with sandbox_executable flag

In [ ]:
from scratch_agent.tools.base import tool

@tool(sandbox_executable=True)
def get_wikipedia_revisions(
    title: str,
    before_date: str,
    limit: int = 10,
    language: str = "en",
    output: str = "full",
) -> list:
    import requests

    api_url = f"https://{language}.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "titles": title,
        "prop": "revisions",
        "rvprop": "ids|timestamp|user|content",
        "rvlimit": limit,
        "rvstart": before_date,
        "rvdir": "older",
        "format": "json",
    }

    response = requests.get(api_url, params=params)
    data = response.json()

    pages = data.get("query", {}).get("pages", {})
    for page_id, page_data in pages.items():
        if page_id == "-1":
            return []
        revisions = page_data.get("revisions", [])
        if output == "content":
            return [rev.get("*", "") for rev in revisions]
        return revisions

    return []

### 8.3.4 Handling sandbox tools in the Agent

**Listing 8.18** Adding sandbox tools list to Agent

In [ ]:
def __init__(self, ...):
    self._sandbox_tools: List[FunctionTool] = []
    self.tools = self._setup_tools(tools or [])

**Listing 8.19** Classifying sandbox_executable tools in `_setup_tools`

*(Reviewer note applied: flat structure with `continue` instead of
nested `if`, and the loop now stacks all incompatible tools so a single
`ValueError` reports them together.)*

In [ ]:
def _setup_tools(self, tools: List[BaseTool]) -> List[BaseTool]:
    tools = list(tools)

    # Identify sandbox_executable tools and stack any errors
    invalid_sandbox_tools = []
    for t in tools:
        if not isinstance(t, FunctionTool) or not t.sandbox_executable:
            continue
        if self.code_execution != "e2b":
            invalid_sandbox_tools.append(t.name)
            continue
        self._sandbox_tools.append(t)

    if invalid_sandbox_tools:
        raise ValueError(
            f"Tools {invalid_sandbox_tools} are marked as sandbox_executable "
            "but code_execution is not enabled."
        )

    # Add code execution tool
    if self.code_execution == "e2b":
        from .tools.code_execution import execute_python
        tools.append(execute_python)

    return tools

**Listing 8.20** Registering sandbox tools during agent execution

In [ ]:
async def run(self, user_input: str, ...):
    # Initialize sandbox
    if self.code_execution == "e2b" and context.code_env is None:
        from e2b_code_interpreter import Sandbox
        context.code_env = Sandbox.create(timeout=300)

        # Register sandbox_executable tools to sandbox
        if self._sandbox_tools:
            self._register_sandbox_tools(context.code_env)

def _register_sandbox_tools(self, sandbox) -> None:
    """Register source code of sandbox_executable tools to sandbox."""
    tool_sources = []
    for t in self._sandbox_tools:
        source = t.get_source_code()
        tool_sources.append(source)

    combined_source = "\n\n".join(tool_sources)
    result = sandbox.run_code(combined_source)

    if result.error:
        raise RuntimeError(f"Failed to register sandbox tools: {result.error}")

**Listing 8.21** Generating system prompt for sandbox-available tools

In [ ]:
def _get_sandbox_tools_prompt(self) -> str:
    """Generate prompt describing tools available in sandbox."""
    if not self._sandbox_tools:
        return ""

    tool_definitions = [t.tool_definition for t in self._sandbox_tools]
    tools_json = json.dumps(tool_definitions, indent=2, ensure_ascii=False)

    return (
        "\n\n## Sandbox-Executable Tools\n"
        "The following functions are pre-registered in the sandbox "
        "and can be called directly in your Python code:\n"
        f"{tools_json}"
    )

### 8.3.5 Practical testing

**Listing 8.22** Setting up the agent with sandbox_executable tools

In [ ]:
# Note: book imports "from scratch_agents"; the runnable package is "scratch_agent".
from scratch_agent import Agent, LlmClient
from scratch_agent.tools.search import get_wikipedia_revisions

agent = Agent(
    model=LlmClient(model="gpt-5"),
    tools=[get_wikipedia_revisions],  # sandbox_executable tool
    instructions="""You can execute Python code to solve problems.
    Use the get_wikipedia_revisions function to fetch Wikipedia page content
    from specific dates.""",
    code_execution="e2b"
)
question = """How many times was a Twitter/X post cited as a reference
on the English Wikipedia pages for each day of August in the last
June 2023 versions of the pages?"""

# result = await agent.run(question, verbose=True)

## 8.4 Workspace: Using file systems and CLI

### 8.4.1 The concept of Workspace

**Listing 8.23** Dynamically generating file search functionality with code

In [ ]:
import glob

error_files = []
for log_file in glob.glob("**/*.log", recursive=True):
    with open(log_file, 'r') as f:
        if 'error' in f.read().lower():
            error_files.append(log_file)

print(f"Found {len(error_files)} files containing 'error': {error_files}")

### 8.4.2 Implementing Workspace tools

**Listing 8.24** Implementing bash_tool for CLI command execution

In [ ]:
from e2b_code_interpreter import Sandbox
from e2b_code_interpreter.models import CommandResult

@tool(name="bash_tool", description="Execute a bash command in a sandboxed environment.")
def bash_tool(context: ExecutionContext, command: str) -> str:
    """Execute a bash command in E2B sandbox."""
    if context.code_env is None:
        raise RuntimeError("No code execution environment available.")

    sandbox: Sandbox = context.code_env
    result = sandbox.commands.run(command)

    return _format_command_result(result)

def _format_command_result(result: CommandResult) -> str:
    """Format E2B CommandResult to readable text."""
    parts = []

    if result.stdout:
        parts.append(f"[stdout]\n{result.stdout}")

    if result.stderr:
        parts.append(f"[stderr]\n{result.stderr}")

    if result.error:
        parts.append(f"[error]\n{result.error}")

    parts.append(f"[exit_code] {result.exit_code}")

    return "\n\n".join(parts) if parts else "[No output]"

**Listing 8.25** Implementing upload_file for file transfer to the sandbox

In [ ]:
import os

@tool(name="upload_file", description="Upload a local file to the E2B sandbox environment.")
def upload_file(
    context: ExecutionContext,
    local_path: str,
    sandbox_path: str | None = None
) -> str:
    """Upload a local file to the E2B sandbox."""
    if context.code_env is None:
        raise RuntimeError("No code execution environment available.")

    sandbox: Sandbox = context.code_env

    if not os.path.exists(local_path):
        return f"Error: Local file not found: {local_path}"

    if not os.path.isfile(local_path):
        return f"Error: Path is not a file: {local_path}"

    if sandbox_path is None:
        filename = os.path.basename(local_path)
        sandbox_path = f"/home/user/{filename}"

    try:
        with open(local_path, "rb") as file:
            sandbox.files.write(sandbox_path, file)

        return f"Successfully uploaded '{local_path}' to '{sandbox_path}'"

    except Exception as e:
        return f"Error uploading file: {str(e)}"

### 8.4.3 Practical example: Analyzing an Excel file

**Listing 8.26** Setting up an agent with Workspace for Excel analysis

In [ ]:
from scratch_agent import Agent, LlmClient

agent = Agent(
    model=LlmClient(model="gpt-5"),
    tools=[],
    instructions="""You have access to a sandboxed Linux environment.
    You can upload files, run bash commands, and execute Python code.
    Use these capabilities to analyze data and solve problems.""",
    code_execution="e2b"
)

prompt = f"""The attached spreadsheet contains the sales of menu items
for a regional fast-food chain. Which city had the greater total sales:
Wharvton or Algrimand?

The file is located at: ./gaia_workspace/7cc4acfa-63fd-4acc-a1a1-e8e529e0a97f.xlsx
"""

# result = await agent.run(prompt, verbose=True)
# print(result.output)

## 8.5 Agent skills

### 8.5.1 The agent skills concept

### 8.5.2 Progressive tool information disclosure

### 8.5.3 Implementation

**Listing 8.27** SkillInfo data class for skill metadata

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class SkillInfo:
    """Class to hold skill information."""
    name: str
    description: str
    path: Path

**Listing 8.28** Parsing YAML frontmatter and loading skill information

In [ ]:
import re
from typing import Optional

def parse_frontmatter(content: str) -> dict:
    """Extract YAML frontmatter from markdown file."""
    pattern = r'^---\s*\n(.*?)\n---'
    match = re.match(pattern, content, re.DOTALL)

    if not match:
        return {}

    result = {}
    for line in match.group(1).split('\n'):
        if ':' in line:
            key, value = line.split(':', 1)
            result[key.strip()] = value.strip().strip('"\'')

    return result

def load_skill(skill_dir: Path) -> Optional[SkillInfo]:
    """Read SKILL.md from skill directory and return SkillInfo."""
    skill_md = skill_dir / "SKILL.md"

    if not skill_md.exists():
        return None

    content = skill_md.read_text(encoding='utf-8')
    frontmatter = parse_frontmatter(content)

    name = frontmatter.get('name')
    description = frontmatter.get('description')

    if not name or not description:
        return None

    return SkillInfo(name=name, description=description, path=skill_dir)

**Listing 8.29** Discovering all skills in a directory

In [ ]:
from typing import List

def discover_skills(skills_path: str | Path) -> List[SkillInfo]:
    """Discover all skills in the skills directory."""
    skills_path = Path(skills_path)

    if not skills_path.exists():
        return []

    skills = []
    for item in skills_path.iterdir():
        if item.is_dir():
            skill = load_skill(item)
            if skill:
                skills.append(skill)

    return sorted(skills, key=lambda s: s.name)

**Listing 8.30** Generating skills section for system prompt

*(Book draft has a typo `if"Skills are installed...` — it should be a
single f-string. The corrected form is shown below.)*

In [ ]:
def generate_skills_prompt(
    skills: List[SkillInfo],
    sandbox_path: str = "/home/user/skills"
) -> str:
    """Generate skill guidance to include in system prompt."""
    if not skills:
        return ""

    lines = [
        "## Available Skills",
        f"Skills are installed at `{sandbox_path}`. "
        "Read the SKILL.md file before using a skill.",
        ""
    ]

    for skill in skills:
        lines.append(f"- **{skill.name}**: {skill.description}")

    lines.append("")
    lines.append("To use a skill:")
    lines.append(f"```bash")
    lines.append(f"cat {sandbox_path}/<skill-name>/SKILL.md")
    lines.append(f"```")

    return "\n".join(lines)

**Listing 8.31** Adding skills_path parameter to Agent constructor

*(Book draft Listing 8.31 contains stray code from Listing 8.30; the
real change is the `__init__` snippet shown below — see
`scratch_agent/agent.py` for the merged version.)*

In [ ]:
class Agent:
    def __init__(
        ...
        skills_path: str | None = None,  # Added
    ):
        ...
        self.skills_path = skills_path

**Listing 8.32** Uploading skills to sandbox during agent execution

In [ ]:
async def run(self, user_input: str, ...):
    if self.code_execution == "e2b" and context.code_env is None:
        context.code_env = Sandbox.create(timeout=300)

        # Upload skill folders to sandbox
        if self._skills and self.skills_path:
            self._upload_skills_to_sandbox(context.code_env)

### 8.5.4 Practical example: PDF merge skill

Within the skills folder, the `pdf-merge` skill has the following structure.

```
skills/
└── pdf-merge/
    └── SKILL.md
```

**Listing 8.33** SKILL.md for PDF merge skill

The skill file is markdown, not Python — so it's shown as a markdown
block. The `pypdf` dependency is assumed to be pre-installed in the
sandbox image; the skill body shows the agent how to use it but does
not handle installation.

```markdown
---
name: pdf-merge
description: Merge multiple PDF files into a single PDF document. Use when the user wants to combine, merge, concatenate, or join PDF files together into one file.
---

# PDF Merge

Merge multiple PDFs into a single document using pypdf.

## Usage
```python
from pypdf import PdfWriter, PdfReader

writer = PdfWriter()
pdf_files = ["file1.pdf", "file2.pdf", "file3.pdf"]

for pdf_file in pdf_files:
    reader = PdfReader(pdf_file)
    for page in reader.pages:
        writer.add_page(page)

with open("merged.pdf", "wb") as output:
    writer.write(output)
```
```

### 8.5.5 Hierarchical tool structure